# Exploratory Data Analysis (EDA) - Frimeet Events,
    Este notebook contiene el análisis exploratorio de datos (EDA) para justificar las decisiones metodológicas de los modelos predictivos de clasificación y regresión del proyecto Frimeet, cumpliendo con la exigencia de que no sea solo decoración, sino el sustento del modelado.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Conectar al Data Warehouse generado por el pipeline ETL
con = duckdb.connect('warehouse.db', read_only=True)
df = con.execute("SELECT * FROM hechos_asistencia_eventos").fetchdf()

# Visualizar las primeras filas para confirmar la carga
df.head()

## 1. Justificación para Clasificación: Desbalance de Clases
Exploramos la variable objetivo `aforo_lleno` para determinar la métrica de evaluación correcta para nuestros modelos de Clasificación (Regresión Logística y K-NN).

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='aforo_lleno', palette='Set2')
plt.title('Distribución de Eventos: Aforo Lleno vs No Lleno')
plt.xlabel('¿Aforo Lleno? (0 = No, 1 = Sí)')
plt.ylabel('Cantidad de Eventos')
plt.show()

print("Distribución porcentual:\n", df['aforo_lleno'].value_counts(normalize=True) * 100)

**Conclusión:** Se observa un claro desbalance en las clases. Por lo tanto, evaluar el modelo de Clasificación usando únicamente el *Accuracy* sería metodológicamente incorrecto. Esta observación justifica nuestra decisión de utilizar el **F1-Score** y el **AUC-ROC** como las métricas principales[cite: 1].

## 2. Justificación para Regresión: Selección de Variables Predictivas
Analizamos la correlación de nuestras features numéricas con la variable objetivo `total_gastado_mxn` para sustentar la regresión lineal.

In [ ]:
# Seleccionamos las variables numéricas clave de nuestro esquema estrella
cols_num = ['num_asistentes_reales', 'duracion_minutos', 'distancia_promedio_km', 'temperatura', 'total_gastado_mxn']

plt.figure(figsize=(8, 6))
sns.heatmap(df[cols_num].corr(), annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title('Mapa de Correlación de Pearson: Features vs Gasto Total')
plt.tight_layout()
plt.show()

**Conclusión:** El mapa de calor demuestra una fuerte correlación lineal positiva entre el número de asistentes, la duración del evento y nuestra variable objetivo (gasto total). Esto justifica la viabilidad y elección de un modelo de **Regresión Lineal** para estimar la derrama económica de la logística de los eventos[cite: 1].

In [ ]:
# Cerrar la conexión al finalizar
con.close()